In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import json
import time
import boto3
from botocore.loaders import Loader

REGION = "us-west-2"
MODEL_S3_URI = "s3://sagemaker-benchmark-xxxxx-model-artifact/gpt-oss-20b/"
ROLE_ARN = "arn:aws:iam::XXXXXXXXXXXX:role/SageMaker-Benchmark-Lumen"
S3_OUTPUT = "s3://sagemaker-benchmark-xxxxx-output/recommendation-output/"
CONFIG_NAME = "ds-rec-config-9"
JOB_NAME = "ds-rec-job-9"

# Load the custom service model into botocore
loader = Loader()
api = json.load(open("./sagemaker-2017-07-24.normal.json"))
session = boto3.Session(region_name=REGION)

# Create client with custom service model
sm = session.client("sagemaker",
                    endpoint_url="https://api.sagemaker.us-west-2.amazonaws.com",
                    api_version=api.get("metadata", {}).get("apiVersion"))

## Lumen setup

In [ ]:
# 1. Create workload config
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "traffic_pattern": {
        "requests_per_second": 10,
        "concurrency": 5,
        "duration_seconds": 300,
    },
    "parameters": {
        "public_dataset": "sharegpt",    # optional, will use synth dataset if omitted
        "prompt_input_tokens_mean": 500,
        "prompt_input_tokens_stddev": 10,
        "output_tokens_mean": 500,
        "output_tokens_stddev": 10,
    },
}

config_resp = sm.create_ai_workload_config(
    AIWorkloadConfigName=CONFIG_NAME,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Config created: {config_resp['AIWorkloadConfigArn']}")

In [ ]:
# 2. Create recommendation job
job_resp = sm.create_ai_recommendation_job(
    AIRecommendationJobName=JOB_NAME,
    ModelSource={
        "S3": {
            "S3Uri": MODEL_S3_URI,
            "ModelAccessConfig": {"AcceptEula": True},
        }
    },
    OutputConfig={"S3OutputLocation": S3_OUTPUT},
    AIWorkloadConfigIdentifier=CONFIG_NAME,
    RoleArn=ROLE_ARN,
    PerformanceTarget={
        "Constraints": [
            {"Metric": "throughput", "Stat": "p90"}, #TTFT and Througput
        ]
    },
    ComputeSpec={
        "InstanceTypes": ["ml.g6.24xlarge"] # Only 1 instance supported for Beta.
    },
    OptimizeModel=False,
    InferenceSpecification = {"Framework": "VLLM"},
)
print(f"Job created: {job_resp['AIRecommendationJobArn']}")

In [ ]:
# 3. Poll until terminal
while True:
    desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)
    status = desc["AIRecommendationJobStatus"]
    print(f"  Status: {status}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)

In [ ]:
# 4. Print recommendations
if status == "Completed":
    recommendations = desc.get("Recommendations", [])
    print(f"\n{len(recommendations)} recommendation(s) found:")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n--- Recommendation {i} ---")
        print(f"  {rec.get('RecommendationDescription', 'N/A')}")
        if rec.get("InstanceDetails"):
            for inst in rec["InstanceDetails"]:
                print(f"  Instance: {inst['InstanceType']} x{inst.get('InstanceCount', 1)}")
        if rec.get("OptimizationDetails"):
            for opt in rec["OptimizationDetails"]:
                print(f"  Optimization: {opt['OptimizationType']}")
        if rec.get("ExpectedPerformance"):
            for perf in rec["ExpectedPerformance"]:
                print(f"  {perf['Metric']}: {perf['Value']} {perf.get('Unit', '')}")
        if rec.get("DeploymentConfiguration"):
            deploy = rec["DeploymentConfiguration"]
            print(f"  Deploy image: {deploy.get('ImageUri', 'N/A')}")
            print(f"  Deploy instance: {deploy.get('InstanceType', 'N/A')}")
else:
    print(f"\nEnded with status: {status}")
    if "FailureReason" in desc:
        print(f"Reason: {desc['FailureReason']}")

In [ ]:
print(json.dumps(desc.get("Recommendations", []), indent=2))

In [ ]:
sm.stop_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
#
# Cleanup
#
sm.delete_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_workload_config(AIWorkloadConfigName=CONFIG_NAME)